## READ ME

* The source data that this notebook is meant to parse and dump into a csv for training originates from the repo which as of 4/18/26 resides in `data/spectra/raw/nist_IR.zip`
  * https://github.com/IvanChernyshov/NistChemData


### Purpose
* the purpose of this file is to parse the jdx files into a structured data format like a csv
  * uses the jcamp library to parse the file

### Output
* jdx_metadata.csv contains data on the compound id, compound name, a few compound properties, data on the spectroscopy equipment and techniques used, and a few descriptive statistics of the spectroscopy data
* jdx_xy_points.csv contains the spectroscopy data as a pair of cordinates (x,y) where x is the wavelength and y is the transmitance
  * logic for parsing such contained in `expand_xy_points`

In [3]:
!pip install jcamp rdkit cirpy

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of jcamp to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 15.1 MB/s eta 0:00:00
  Created wheel for jcamp: filename=jcamp-1.2.2-py2.py3-none-any.whl size=10652 sha256=89bde51009a38678a71e1541be82666ffb51c356d3f6de07547eaa6a395df0fa
  Stored in directory: /root/.cache/pip/wheels/6a/d1/9f/9599e61ea4096b380977b3e343d144d1705615c3608f9c034d
  Created wheel for cirpy: filename=CIRpy-1.0.2-py3-none-any.whl size=7263 sha256=68d7b4c497fcaa8058ce7dcf1de180269ae722050178a9d9f9947ef43013e007
  Stored in directory: /root/.cache/pip/wheels/af/e9

In [2]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
from jcamp import jcamp_read

import cirpy
import json
import time
from rdkit import Chem
from urllib.parse import quote
from urllib.request import urlopen
from typing import Any



## Parse JDX into csv

## funcitons to parse jdx using jcamp

In [ ]:
def as_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, np.ndarray):
        return ""
    if isinstance(value, (np.integer, np.floating)):
        return str(value.item())
    return str(value).replace("\r\n", "\n").replace("\r", "\n").replace("\n", " ").strip()


def as_number(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (np.integer, np.floating)):
        return str(value.item())
    if isinstance(value, (int, float)):
        return str(value)
    return as_text(value)


def expand_xy_points(data: dict[str, Any]) -> pd.DataFrame:
    """Return explicit x/y pairs for the spectrum.

    JCAMP files often encode an IR trace as a starting x value plus a delta x,
    followed by a list of y values. For example, a line like:

        549.759 29.3 29.05 28.32 ...

    means:
        (549.759, 29.3)
        (549.759 + deltax * 1, 29.05)
        (549.759 + deltax * 2, 28.32)

    The `jcamp` reader usually gives us explicit `x` and `y` arrays already.
    If those arrays are missing, we rebuild x from `firstx` and `deltax`.
    """

    x = data.get("x")
    y = data.get("y")

    if x is not None and y is not None and len(x) == len(y) and len(x) > 0:
        return pd.DataFrame({"x": np.asarray(x, dtype=float), "y": np.asarray(y, dtype=float)})

    if y is None or len(y) == 0:
        return pd.DataFrame(columns=["x", "y"])

    y_values = np.asarray(y, dtype=float)
    firstx = data.get("firstx")
    deltax = data.get("deltax")

    if firstx is not None and deltax is not None:
        x_values = float(firstx) + float(deltax) * np.arange(len(y_values))
    else:
        x_values = np.arange(len(y_values), dtype=float)

    return pd.DataFrame({"x": x_values, "y": y_values})


def parse_jdx_file(path: Path) -> tuple[dict[str, str], pd.DataFrame]:
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        data = jcamp_read(handle)

    metadata: dict[str, str] = {"filename": path.name}
    for key, value in data.items():
        if key in {"x", "y"}:
            continue
        if isinstance(value, np.ndarray):
            continue
        metadata[key] = as_text(value)

    x = data.get("x")
    y = data.get("y")
    metadata["npoints"] = as_number(data.get("npoints", len(x) if x is not None else ""))
    metadata["x_min"] = as_number(np.min(x) if x is not None and len(x) else "")
    metadata["x_max"] = as_number(np.max(x) if x is not None and len(x) else "")
    metadata["y_min"] = as_number(np.min(y) if y is not None and len(y) else "")
    metadata["y_max"] = as_number(np.max(y) if y is not None and len(y) else "")
    metadata["firstx"] = as_number(data.get("firstx"))
    metadata["lastx"] = as_number(data.get("lastx"))
    metadata["deltax"] = as_number(data.get("deltax"))

    spectrum = expand_xy_points(data)
    if not spectrum.empty:
        spectrum.insert(0, "filename", path.name)
        spectrum.insert(1, "point_index", np.arange(len(spectrum), dtype=int))
    else:
        spectrum = pd.DataFrame(columns=["filename", "point_index", "x", "y"])

    return metadata, spectrum


### Get list of jdx files


In [ ]:
jdx_dir = Path.cwd() / "data" / "IR"
if not jdx_dir.exists():
    alt_dir = Path.cwd().parent / "data" / "IR"
    if alt_dir.exists():
        jdx_dir = alt_dir

jdx_files = sorted(jdx_dir.glob("*.jdx"))
print("jdx dir:", jdx_dir)
print("jdx files are:", len(jdx_files))
jdx_files[:5]


jdx dir: /home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR
jdx files are: 19582


[PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000033_IR_0.jdx'),
 PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000034_IR_0.jdx'),
 PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000036_IR_0.jdx'),
 PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000043_IR_0.jdx'),
 PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000044_IR_0.jdx')]

### parse each file and concatonate into csv

In [ ]:
metadata_rows: list[dict[str, str]] = []
xy_frames: list[pd.DataFrame] = []
failed_files: list[dict[str, str]] = []

for path in jdx_files:
    try:
        metadata, spectrum = parse_jdx_file(path)
        metadata_rows.append(metadata)
        xy_frames.append(spectrum)
    except Exception as exc:
        failed_files.append({"filename": path.name, "error": f"{type(exc).__name__}: {exc}"})

metadata_df = pd.DataFrame(metadata_rows)
xy_df = pd.concat(xy_frames, ignore_index=True) if xy_frames else pd.DataFrame(columns=["filename", "point_index", "x", "y"])
failed_df = pd.DataFrame(failed_files)

metadata_df.head()


,filename,title,jcamp-dx,data type,class,origin,owner,date,names,molform,...,sphere diameter,acquisition mode,coadded scans,phase resolution,zerofilling,spectral resolution,wavenumber accuracy,apodization function,low pass filter,switch gain on
0,B6000033_IR_0.jdx,"ETHANOL, 2,2'-(5-CHLORO-2-ETHOXY PHENYLIMIDO) DI",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","ANILINE, 2-ETHOXY-5-CHLORO-N,N-(2,2'-DIETHANOL)",C12 H18 Cl N O3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B6000034_IR_0.jdx,2-(O-CHLOROPHENYL)BENZOXAZONE-4,4.24,INFRARED SPECTRUM,COBLENTZ,NaN,COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","2-(2-chlorophenyl)-2,3-dihydro-4H-1,3-benzoxaz...",C14 H10 N O2 Cl,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B6000036_IR_0.jdx,2-ISOBUTYLBENZOXAZONE-4,4.24,INFRARED SPECTRUM,COBLENTZ,NaN,COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","2-isobutyl-2,3-dihydro-4H-1,3-benzoxazin-4-one",C12 H15 N O2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,B6000043_IR_0.jdx,"SULFONYL-O,O'-DIPHENACYLDIBENZOATE",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970",2-[3-((3-[(benzoyloxy)acetyl]phenyl)sulfonyl)p...,C30 H22 O8 S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,B6000044_IR_0.jdx,"SULFONYL-O,M'-DIBENZOIC ACID, DIPHENACYL ESTER",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970",NaN,C30 H22 O8 S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
xy_df.head()


,filename,point_index,x,y
0,B6000033_IR_0.jdx,0,2.035,0.751
1,B6000033_IR_0.jdx,1,2.041613,0.751
2,B6000033_IR_0.jdx,2,2.048225,0.751
3,B6000033_IR_0.jdx,3,2.054838,0.751
4,B6000033_IR_0.jdx,4,2.06145,0.751


### Output metadata and coordinates into csvs

In [ ]:
metadata_out = Path("data/jdx_metadata.csv")
xy_out = Path("data/jdx_xy_points.csv")

metadata_out.parent.mkdir(parents=True, exist_ok=True)
metadata_df.to_csv(metadata_out, index=False)
xy_df.to_csv(xy_out, index=False)

print(f"Wrote {len(metadata_df)} metadata rows to {metadata_out}")
print(f"Wrote {len(xy_df)} xy rows to {xy_out}")

if not failed_df.empty:
    display(failed_df)


Wrote 19581 metadata rows to data/jdx_metadata.csv
Wrote 59986786 xy rows to data/jdx_xy_points.csv


,filename,error
0,B6000446_IR_0.jdx,Exception: Unknown character (t) encountered w...


## Normalize compound id

In [ ]:

i = 0
_smiles_cache: dict = {}

try:
    with open("smiles_cache.json") as f:
        _smiles_cache = json.load(f)
except:
    pass

def save_cache():
    with open("smiles_cache.json", "w") as f:
        json.dump(_smiles_cache, f)

def as_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()

def normalize_cas(value: Any) -> str:
    text = as_text(value)
    if not text:
        return ""
    match = next((chunk for chunk in text.replace(";", " ").replace(",", " ").split() if chunk.count("-") == 2), "")
    return match.strip() if match else text.strip()

def is_valid_smiles(smiles: Any) -> bool:
    text = as_text(smiles)
    if not text:
        return False
    if Chem is None:
        return True
    return Chem.MolFromSmiles(text) is not None

def resolve_smiles_with_cirpy(identifier: str) -> str:
    if not identifier or cirpy is None:
        return ""
    try:
        result = cirpy.resolve(identifier, "smiles")
    except Exception:
        return ""
    return as_text(result)

def resolve_smiles_with_pubchem(identifier: str) -> str:
    if not identifier:
        return ""
    if identifier in _smiles_cache:
        return _smiles_cache[identifier]
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{quote(identifier, safe='')}/property/CanonicalSMILES/JSON"
    try:
        with urlopen(url, timeout=20) as r:
            data = json.loads(r.read())
            smiles = data["PropertyTable"]["Properties"][0]["CanonicalSMILES"]
            _smiles_cache[identifier] = smiles
            time.sleep(0.2)  # stay under PubChem's 5 req/sec limit
            return smiles
    except Exception:
        _smiles_cache[identifier] = ""
        return ""

def resolve_smiles_from_row(row: pd.Series) -> tuple[str, str, str]:
    global i
    i += 1
    if i % 100 == 0:
        save_cache()
        print(f"reached row {i}, cache size: {len(_smiles_cache)}")

    candidates = [
        ("cas registry no", normalize_cas(row.get("cas registry no"))),
        ("cas_registry_no", normalize_cas(row.get("cas_registry_no"))),
        ("title", as_text(row.get("title"))),
        ("names", as_text(row.get("names"))),
        ("name", as_text(row.get("name"))),
    ]

    for field, identifier in candidates:
        if not identifier:
            continue
        smiles = resolve_smiles_with_cirpy(identifier)
        if is_valid_smiles(smiles):
            return smiles, f"cirpy:{field}", identifier
        smiles = resolve_smiles_with_pubchem(identifier)
        if is_valid_smiles(smiles):
            return smiles, f"pubchem:{field}", identifier

    return "", "", ""

In [ ]:
smiles_values = metadata_df.apply(resolve_smiles_from_row, axis=1, result_type="expand")
smiles_values.columns = ["smiles", "smiles_method", "smiles_query"]
metadata_df = pd.concat([metadata_df, smiles_values], axis=1)
smiles_df = metadata_df.loc[metadata_df["smiles"].map(is_valid_smiles)].copy()


reached row 100, cache size: 0
reached row 200, cache size: 0
reached row 300, cache size: 0
reached row 400, cache size: 0
reached row 500, cache size: 0
reached row 600, cache size: 0
reached row 700, cache size: 0
reached row 800, cache size: 0
reached row 900, cache size: 0
reached row 1000, cache size: 0
reached row 1100, cache size: 0
reached row 1200, cache size: 0
reached row 1300, cache size: 0
reached row 1400, cache size: 0


[08:24:15] WARNING: not removing hydrogen atom without neighbors
[08:24:15] WARNING: not removing hydrogen atom without neighbors
[08:24:18] Explicit valence for atom # 1 O, 4, is greater than permitted
[08:24:21] WARNING: not removing hydrogen atom without neighbors
[08:24:21] Explicit valence for atom # 5 N, 4, is greater than permitted


reached row 1500, cache size: 15
reached row 1600, cache size: 22


[08:24:41] WARNING: not removing hydrogen atom without neighbors
[08:24:47] WARNING: not removing hydrogen atom without neighbors


reached row 1700, cache size: 48


...

[10:35:06] WARNING: not removing hydrogen atom without neighbors


reached row 18800, cache size: 2365
reached row 18900, cache size: 2389
reached row 19000, cache size: 2411
reached row 19100, cache size: 2444
reached row 19200, cache size: 2462
reached row 19300, cache size: 2484


[10:39:21] WARNING: not removing hydrogen atom without neighbors
[10:39:33] WARNING: not removing hydrogen atom without neighbors


reached row 19400, cache size: 2512
reached row 19500, cache size: 2539


[10:41:28] WARNING: not removing hydrogen atom without neighbors
[10:41:28] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not removing hydrogen atom without neighbors
[10:41:29] WARNING: not r

## Join spectroscopy metadata with xy points  

In [4]:
smiles_df = pd.read_csv("/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR spectroscopy data/output.csv")
xy_df = pd.read_csv("/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR spectroscopy data/jdx_xy_points.csv")

/tmp/ipykernel_81847/1595487314.py:1: DtypeWarning: Columns (0: jcamp-dx, 1: $nist id, 2: number of interferograms averaged per single channel spectrum) have mixed types. Specify dtype option on import or set low_memory=False.
  smiles_df = pd.read_csv("/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR spectroscopy data/output.csv")


In [5]:
print("before join smiles df is ", smiles_df.shape)
print("before join xy df is ", xy_df.shape)

before join smiles df is  (19565, 81)
before join xy df is  (59986786, 4)


In [6]:
# Perform the join operation
joined_df = pd.merge(smiles_df, xy_df, on="filename", how="inner")

print("Joined DataFrame head:")
display(joined_df.head())

Joined DataFrame head:


,filename,title,jcamp-dx,data type,class,origin,owner,date,names,molform,...,wavenumber accuracy,apodization function,low pass filter,switch gain on,smiles,smiles_method,smiles_query,point_index,x,y
0,B6000033_IR_0.jdx,"ETHANOL, 2,2'-(5-CHLORO-2-ETHOXY PHENYLIMIDO) DI",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","ANILINE, 2-ETHOXY-5-CHLORO-N,N-(2,2'-DIETHANOL)",C12 H18 Cl N O3,...,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,0,2.035000,0.751
1,B6000033_IR_0.jdx,"ETHANOL, 2,2'-(5-CHLORO-2-ETHOXY PHENYLIMIDO) DI",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","ANILINE, 2-ETHOXY-5-CHLORO-N,N-(2,2'-DIETHANOL)",C12 H18 Cl N O3,...,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,1,2.041613,0.751
2,B6000033_IR_0.jdx,"ETHANOL, 2,2'-(5-CHLORO-2-ETHOXY PHENYLIMIDO) DI",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","ANILINE, 2-ETHOXY-5-CHLORO-N,N-(2,2'-DIETHANOL)",C12 H18 Cl N O3,...,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,2,2.048225,0.751
3,B6000033_IR_0.jdx,"ETHANOL, 2,2'-(5-CHLORO-2-ETHOXY PHENYLIMIDO) DI",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","ANILINE, 2-ETHOXY-5-CHLORO-N,N-(2,2'-DIETHANOL)",C12 H18 Cl N O3,...,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,3,2.054838,0.751
4,B6000033_IR_0.jdx,"ETHANOL, 2,2'-(5-CHLORO-2-ETHOXY PHENYLIMIDO) DI",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","ANILINE, 2-ETHOXY-5-CHLORO-N,N-(2,2'-DIETHANOL)",C12 H18 Cl N O3,...,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,4,2.061450,0.751


In [8]:
aggregated_xy_df = xy_df.groupby('filename').agg(x_coords=('x', list), y_coords=('y', list)).reset_index()

In [9]:
metadata_with_aggregated_xy_df = pd.merge(smiles_df, aggregated_xy_df, on='filename', how='inner')

print("DataFrame with aggregated XY coordinates head:")
display(metadata_with_aggregated_xy_df.head())

DataFrame with aggregated XY coordinates head:


,filename,title,jcamp-dx,data type,class,origin,owner,date,names,molform,...,spectral resolution,wavenumber accuracy,apodization function,low pass filter,switch gain on,smiles,smiles_method,smiles_query,x_coords,y_coords
0,B6000033_IR_0.jdx,"ETHANOL, 2,2'-(5-CHLORO-2-ETHOXY PHENYLIMIDO) DI",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","ANILINE, 2-ETHOXY-5-CHLORO-N,N-(2,2'-DIETHANOL)",C12 H18 Cl N O3,...,NaN,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,"[2.035, 2.0416126, 2.0482252, 2.0548378, 2.061...","[0.751, 0.751, 0.751, 0.751, 0.751, 0.751, 0.7..."
1,B6000034_IR_0.jdx,2-(O-CHLOROPHENYL)BENZOXAZONE-4,4.24,INFRARED SPECTRUM,COBLENTZ,NaN,COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","2-(2-chlorophenyl)-2,3-dihydro-4H-1,3-benzoxaz...",C14 H10 N O2 Cl,...,NaN,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,"[2.031, 2.0381808, 2.0453616, 2.0525424, 2.059...","[0.982, 0.982, 0.982, 0.982, 0.982, 0.982, 0.9..."
2,B6000036_IR_0.jdx,2-ISOBUTYLBENZOXAZONE-4,4.24,INFRARED SPECTRUM,COBLENTZ,NaN,COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","2-isobutyl-2,3-dihydro-4H-1,3-benzoxazin-4-one",C12 H15 N O2,...,NaN,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,"[2.042, 2.049153, 2.056306, 2.063459, 2.070612...","[0.98, 0.98, 0.98, 0.978, 0.978, 0.978, 0.975,..."
3,B6000043_IR_0.jdx,"SULFONYL-O,O'-DIPHENACYLDIBENZOATE",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970",2-[3-((3-[(benzoyloxy)acetyl]phenyl)sulfonyl)p...,C30 H22 O8 S,...,NaN,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,"[2.022, 2.0284796, 2.0349592, 2.0414388, 2.047...","[0.637, 0.637, 0.639, 0.639, 0.639, 0.639, 0.6..."
4,B6000044_IR_0.jdx,"SULFONYL-O,M'-DIBENZOIC ACID, DIPHENACYL ESTER",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970",NaN,C30 H22 O8 S,...,NaN,NaN,NaN,NaN,NaN,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,cirpy:cas registry no,NaN,"[2.023, 2.0294712, 2.0359424, 2.0424136, 2.048...","[0.564, 0.564, 0.564, 0.564, 0.564, 0.564, 0.5..."


In [10]:
print(f"New DataFrame shape: {metadata_with_aggregated_xy_df.shape}")

New DataFrame shape: (19213, 83)


In [ ]:
metadata_with_aggregated_xy_df.to_csv("data/IR spectroscopy data/NIST_IR_Spectroscopy.csv", index=False)
